In [37]:
#CNN Image classification on CIFAR_10
##Student Id: 23-52506-2


##Import Libraries

In [38]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
print("Libraries has been imported successfully")

Libraries has been imported successfully


### Data Preprocessing
In this step, the images are converted into tensors and normalized.
Normalization helps the model learn faster and improves performance.

In [39]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

## Dataset Loading
The CIFAR-10 dataset is loaded using PyTorch.
It contains 10 different image classes such as airplane, cat, dog, etc.

In [40]:
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

print(f"Training data size: {len(train_data)}")
print(f"Test data size: {len(test_data)}")

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
print(f"Classes: {classes}")

Training data size: 50000
Test data size: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Train-Validation Split
The dataset is divided into training and validation sets.
80% is used for training and 20% for validation.

In [41]:
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size

train_dataset, val_dataset = random_split(train_data, [train_size, val_size])

print(f"Training samples: {train_size}")
print(f"Validation samples: {val_size}")

Training samples: 40000
Validation samples: 10000


In [42]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

## CNN Model
A Convolutional Neural Network (CNN) is defined.
It consists of convolution layers, pooling, and fully connected layers for classification.

In [43]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(-1, 32 * 8 * 8)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN().to(device)

print(f"Model is running on: {device}")

Model is running on: cpu


In [44]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

## Training Process
The model is trained using cross-entropy loss and Adam optimizer.
Accuracy and loss are calculated for both training and validation data.

In [ ]:
train_losses = []
val_losses = []
train_acc = []
val_acc = []

num_epochs = 5

print("Starting training...")
print("-" * 50)

for epoch in range(num_epochs):
    model.train()
    correct = 0
    total = 0
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_losses.append(running_loss / len(train_loader))
    train_acc.append(100 * correct / total)

    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_losses.append(val_loss / len(val_loader))
    val_acc.append(100 * correct / total)

    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_losses[-1]:.4f}, Train Acc: {train_acc[-1]:.2f}%")
    print(f"  Val Loss: {val_losses[-1]:.4f}, Val Acc: {val_acc[-1]:.2f}%")
    print("-" * 30)

print("Training completed!")

Starting training...
--------------------------------------------------
Epoch 1/5
  Train Loss: 1.6086, Train Acc: 41.28%
  Val Loss: 1.2711, Val Acc: 54.23%
------------------------------
Epoch 2/5
  Train Loss: 1.3414, Train Acc: 51.61%
  Val Loss: 1.1673, Val Acc: 58.46%
------------------------------
Epoch 3/5
  Train Loss: 1.2301, Train Acc: 56.13%
  Val Loss: 1.0502, Val Acc: 62.96%
------------------------------
Epoch 4/5
  Train Loss: 1.1067, Train Acc: 60.70%
  Val Loss: 1.0008, Val Acc: 64.12%
------------------------------


## Model Evaluation
After training, the model is tested on unseen data to measure its accuracy.

In [ ]:
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = 100 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")

## Classification Report
Precision, recall, and F1-score are calculated for each class
to evaluate the model performance in detail.

In [ ]:
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))

## Confusion Matrix
The confusion matrix shows how well the model predicts each class.
It helps identify correct and incorrect predictions.

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)
print("\nPer-class Accuracy:")
for i, class_name in enumerate(classes):
    print(f"{class_name:12s}: {per_class_acc[i]:.2%}")

best_class = classes[np.argmax(per_class_acc)]
worst_class = classes[np.argmin(per_class_acc)]
print(f"\nBest performing class: {best_class}")
print(f"Worst performing class: {worst_class}")

## Training Curves
Graphs are plotted to visualize training and validation accuracy and loss over epochs.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_acc, label='Train Accuracy', marker='o')
plt.plot(val_acc, label='Validation Accuracy', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
torch.save(model.state_dict(), "cnn_model.pth")
print("Model saved as 'cnn_model.pth'")

## Conclusion
The CNN model was successfully trained on the CIFAR-10 dataset.
The model achieved reasonable accuracy and performance.
Further improvements can be made by increasing epochs or tuning the model.

In [ ]:
# Download the saved model from Colab
from google.colab import files
files.download('cnn_model.pth')
print("Model file download started!")